# Multihead Displaced Queries

A tiny Flower-style block: one branch does point-wise local affine mixing. other branch predicts per-head offsets, sample matching-head values, mix the result.

![Flowers point cloud input](flowers.png)

In [ ]:
import torch
import torch.nn as nn
import torchbvh as tb

class Model(nn.Module):
    def __init__(self, in_channels, n_heads, spatial_dim, out_channels, k=4):
        super().__init__()
        self.n_heads = n_heads
        self.spatial_dim = spatial_dim
        self.k = k

        # learnable projections
        self.displacement_head = nn.Linear(in_channels, n_heads * spatial_dim)
        self.value_head = nn.Linear(in_channels, n_heads * in_channels)
        self.mix = nn.Linear(n_heads * in_channels, out_channels)


    def forward(self, pos, x):
        """
        Args:
            pos: (B, N, D) source positions
            x: (B, N, C) source features
        Returns:
            out: (B, N, out_channels) interpolated features
        """

        B, N, D = pos.shape
        C = x.shape[-1]
        H = self.n_heads

        offsets = torch.tanh(self.displacement_head(x)).reshape(B, N, H, D)
        values = self.value_head(x).reshape(B, N, H, C)
        queries = pos[:, :, None, :] + offsets

        # interpolate_displaced accepts per-point, per-head queries: (B, N, H, D)
        # and samples from matching per-head values: (B, N, H, C).
        sampled = tb.interpolate_displaced(pos, queries, values, k=self.k)
        out = self.mix(sampled.reshape(B, N, H * C))
        return out


# dummy data
B, N, H, D, C = 16, 1000, 40, 3, 8
pos = torch.randn(B, N, D, device="cuda")
x = torch.randn(B, N, C, device="cuda", requires_grad=True)

# example loop
model = Model(C, H, D, C).cuda()
out = model(pos, x)
loss = out.square().mean() # dummy loss
loss.backward()
print("loss:", loss.item())